In [ ]:
import os
import pandas as pd
import numpy as np
import networkx as nx
import warnings
import time
from pathlib import Path
from gurobipy import Model, GRB, quicksum
import scipy.stats as st
warnings.filterwarnings('ignore')

In [ ]:
# PATHS, CONSTANTS, AND SIM PARAMS

BASE = Path(os.environ.get("KEP_DATA_DIR", "../../data"))
POOL_DIR     = BASE / 'pool_simulations'
MATRICES_DIR = BASE / 'pool_matrices'
RESULTS_DIR  = BASE / 'supplementary' / 'simulation_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_SIMS = 100
LOCI = ['A', 'B', 'C', 'DR', 'DQ']
EPLET_CLASSES = ['ClassI', 'DR', 'DQ']
ETHCATS = [1, 2, 4, 5, 6, 7]
TESTED_ETHCATS = [1, 2, 4, 5]                
ETH_LABELS = {1: 'Caucasian', 2: 'Afroamerican', 4: 'Latin', 5: 'Asian',
              6: 'AmInd', 7: 'PacIsl'}


df_pat = pd.read_csv(BASE / 'df_receptores_imputados_final.csv', low_memory=False)
df_pat['WL_ID_CODE'] = df_pat['WL_ID_CODE'].astype('int64')
df_pat['ETHCAT'] = pd.to_numeric(df_pat['ETHCAT'], errors='coerce')
print(f'Loaded {len(df_pat)} patient demographics')

SIM_PARAMS = {
    'TOTAL_TIME':       10 * 12,
    'ARRIVAL_RATE':     1000 / (10 * 12),
    'MEAN_PATIENCE':    65.1552,   
    'MATCH_RUN':        3,
    'WARMUP_MONTHS':    60,   # warmup
    'MAX_CYCLE_LENGTH': 3,
    'SEED_BASE':        42,
    'P':                1100,       
    'k_opt':            0,           
}


PERTURBATIONS = {'min': 0.9999, 'max': 1.0001}

print('Setup:')
print(f"  ARRIVAL_RATE:       {SIM_PARAMS['ARRIVAL_RATE']:.4f} /mo")
print(f"  MATCH_RUN:          every {SIM_PARAMS['MATCH_RUN']} mo")
print(f"  MAX_CYCLE_LENGTH:   {SIM_PARAMS['MAX_CYCLE_LENGTH']}")
print(f"  Tested ethnicities: {[ETH_LABELS[e] for e in TESTED_ETHCATS]}")
print(f"  Perturbations:      {PERTURBATIONS}")
print(f"  Total runs:         {len(TESTED_ETHCATS) * len(PERTURBATIONS)} (4 eth × 2 directions)")

In [ ]:
# PER SIM DATA LOADER 

def load_sim_data(sim_id):
    pool_df = pd.read_parquet(POOL_DIR / f'pairs_sim_{sim_id:03d}.parquet')
    sim_dir = MATRICES_DIR / f'sim_{sim_id:03d}'
    compat = pd.read_parquet(sim_dir / 'compatibility.parquet').values.astype(np.int8)
    antigen_mm = {L: pd.read_parquet(sim_dir / f'mismatch_antigen_{L}.parquet').values for L in LOCI}
    allele_mm  = {L: pd.read_parquet(sim_dir / f'mismatch_allele_{L}.parquet').values for L in LOCI}
    eplet_mm   = {cls: pd.read_parquet(sim_dir / f'mismatch_eplet_{cls}.parquet').values for cls in EPLET_CLASSES}
    return {'pool_df': pool_df, 'compat': compat,
            'antigen_mm': antigen_mm, 'allele_mm': allele_mm, 'eplet_mm': eplet_mm}

In [ ]:
# BUILD WEIGHT MATRICES 

MAX_ANTIGEN_10LOCI = 10
MAX_ALLELE_10LOCI  = 10
MAX_EPLET_10LOCI   = 140

def build_weights_10loci(sim_data):
    am = sim_data['antigen_mm']; al = sim_data['allele_mm']; ep = sim_data['eplet_mm']
    sum_antigen = sum(am[L] for L in LOCI).astype(np.int32)
    sum_allele  = sum(al[L] for L in LOCI).astype(np.int32)
    sum_eplet   = (ep['ClassI'] + ep['DR'] + ep['DQ']).astype(np.int32)
    return {
        'antigen': (MAX_ANTIGEN_10LOCI - sum_antigen).astype(np.int32),
        'allele':  (MAX_ALLELE_10LOCI  - sum_allele).astype(np.int32),
        'eplet':   (MAX_EPLET_10LOCI   - sum_eplet).astype(np.int32),
        'score_classI': (6 - (am['A'] + am['B'] + am['C'])).astype(np.int32),
        'score_DR':     (2 - am['DR']).astype(np.int32),
        'score_DQ':     (2 - am['DQ']).astype(np.int32),
    }

In [ ]:
# GRAPH (ABO+DSA only)

def create_graph(waiting_indices, compat):
    G = nx.DiGraph()
    G.add_nodes_from(waiting_indices)
    for i in waiting_indices:
        for j in waiting_indices:
            if i != j and compat[i, j] == 1:
                G.add_edge(j, i)
    return G

In [ ]:
# MAXIMUM TRANSPLANT OBJECTIVE WITH ETHNICITY MULTIPLIERS (NO HLA TERM)
#
# Cycle weight (manuscript Eq. 13 with HLA part removed):
#     w_C^{t_j} = Σ_{arcs (u,v) in C}  v( s(r_v) )


def optimization_max_transplant_weighted(G, pair_ethcat, multipliers, l=3):
    total_cycles = list(nx.simple_cycles(G, length_bound=l))
    if not total_cycles:
        return nx.DiGraph(), []

    m = Model('maxtx_weighted')
    m.setParam('OutputFlag', 0)
    x = {tuple(c): m.addVar(vtype=GRB.BINARY) for c in total_cycles}

    def cycle_payoff(c):
        return sum(
            multipliers.get(int(pair_ethcat[v]), 1.0)
            for _, v in zip(c, c[1:] + c[:1])
        )

    m.setObjective(quicksum(x[tuple(c)] * cycle_payoff(c) for c in total_cycles), GRB.MAXIMIZE)
    for node in G.nodes():
        m.addConstr(quicksum(x[tuple(c)] for c in total_cycles if node in c) <= 1)
    m.optimize()

    G_opt = nx.DiGraph()
    selected = []
    if m.status == GRB.OPTIMAL:
        for c in total_cycles:
            if x[tuple(c)].X > 0.5:
                selected.append(c)
                for i in range(len(c)):
                    u, v = c[i], c[(i + 1) % len(c)]
                    G_opt.add_edge(u, v)
    return G_opt, selected

In [ ]:
# RUN ONE SIMULATION WITH MULTIPLIERS  (MaxTransplant objective)


def run_simulation_maxtx(sim_id, compat, weights, pair_ethcat, multipliers, params):
    n = compat.shape[0]
    ss = np.random.SeedSequence(params['SEED_BASE'] + sim_id * 1000)
    rng_arr, rng_dep = (np.random.default_rng(s) for s in ss.spawn(2))

    available = set(range(n))
    waiting = []
    arrival_t, departure_t = {}, {}
    historial_cycles = []
    historial_departures = []
    pool_sizes = []
    deadline = {}
    runs_participated = {}
    pool_sizes_by_eth = {e: [] for e in ETHCATS}
    arrivals_by_eth   = {e: 0 for e in ETHCATS}
    departures_by_eth = {e: 0 for e in ETHCATS}

    quality = {(res, e): [] for res in ('antigen', 'allele', 'eplet') for e in ETHCATS}
    quality.update({(cls, e): [] for cls in ('classI', 'DR', 'DQ') for e in ETHCATS})

    WARMUP = params.get('WARMUP_MONTHS', 0)
    for month in range(params['TOTAL_TIME']):
        counting = month >= WARMUP
        
        n_arr = rng_arr.poisson(params['ARRIVAL_RATE'])
        if n_arr > 0:
            new_pairs = rng_arr.choice(list(available), size=n_arr, replace=False)
            for p in new_pairs:
                p_int = int(p)
                arrival_t[p_int] = month
                available.discard(p_int)
                waiting.append(p_int)
                deadline[p_int] = month + rng_dep.exponential(params['MEAN_PATIENCE'])
                e = int(pair_ethcat[p_int])
                if counting and e in arrivals_by_eth: arrivals_by_eth[e] += 1

   
        if (month + 1) % params['MATCH_RUN'] == 0 and len(waiting) >= 2:
            if counting: pool_sizes.append(len(waiting))
            for e_ps in (ETHCATS if counting else []):
                pool_sizes_by_eth[e_ps].append(sum(1 for w in waiting if int(pair_ethcat[w]) == e_ps))
            for _w in waiting:
                runs_participated[_w] = runs_participated.get(_w, 0) + 1
            G = create_graph(waiting, compat)
            G_opt, selected = optimization_max_transplant_weighted(
                G, pair_ethcat, multipliers, l=params['MAX_CYCLE_LENGTH'])

            for u, v in (G_opt.edges() if counting else []):
                e = int(pair_ethcat[v])
                if e not in arrivals_by_eth: continue
                quality[('antigen', e)].append(int(weights['antigen'][v, u]))
                quality[('allele',  e)].append(int(weights['allele'][v, u]))
                quality[('eplet',   e)].append(int(weights['eplet'][v, u]))
                quality[('classI',  e)].append(int(weights['score_classI'][v, u]))
                quality[('DR',      e)].append(int(weights['score_DR'][v, u]))
                quality[('DQ',      e)].append(int(weights['score_DQ'][v, u]))

            historial_cycles.extend(selected if counting else [])
            cycled = {p for c in selected for p in c}
            waiting = [w for w in waiting if w not in cycled]
            for p_int in cycled: departure_t[int(p_int)] = month

        departed_now = [w for w in waiting if deadline[w] <= month]
        if departed_now:
            ds = set(departed_now)
            waiting = [w for w in waiting if w not in ds]
            for p_int in (departed_now if counting else []):
                historial_departures.append(p_int)
                e = int(pair_ethcat[p_int])
                if e in departures_by_eth:
                    departures_by_eth[e] += 1

    waiting_times_by_eth = {e: [] for e in ETHCATS}
    for p in {p for c in historial_cycles for p in c}:
        if p in runs_participated:
            e = int(pair_ethcat[p])
            if e in waiting_times_by_eth:
                waiting_times_by_eth[e].append(runs_participated[p])

    n_total_arr = sum(arrivals_by_eth.values())
    n_total_tx  = sum(len(c) for c in historial_cycles)
    F_total = n_total_tx / max(n_total_arr, 1)
    L_total = len(historial_departures) / max(n_total_arr, 1)
    F_per_eth = {e: sum(1 for c in historial_cycles for p in c if int(pair_ethcat[p]) == e)
                       / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}
    L_per_eth = {e: departures_by_eth[e] / max(arrivals_by_eth.get(e, 0), 1) for e in ETHCATS}

    return {
        'sim_id': sim_id,
        'total_arrivals': n_total_arr, 'total_transplants': n_total_tx,
        'total_departures': len(historial_departures),
        'arrivals_by_eth': arrivals_by_eth, 'departures_by_eth': departures_by_eth,
        'F_per_eth': F_per_eth, 'L_per_eth': L_per_eth,
        'F_total': F_total, 'L_total': L_total,
        'quality': quality, 'waiting_times_by_eth': waiting_times_by_eth,
        'historial_cycles': historial_cycles,
        'avg_pool_size': float(np.mean(pool_sizes)) if pool_sizes else 0.0,
        'avg_pool_size_by_eth': {e: float(np.mean(pool_sizes_by_eth[e])) if pool_sizes_by_eth[e] else 0.0 for e in ETHCATS},
    }

In [ ]:
# PRELOAD: all 100 sims (compat + weights + pair_ethcat)

print('Preloading 100 sims into memory (one-time cost)...')
t0 = time.time()
all_compat = []
all_weights = []
all_pair_ethcat = []
for sim_id in range(N_SIMS):
    sd = load_sim_data(sim_id)
    ws = build_weights_10loci(sd)
    eth_join = sd['pool_df'].merge(df_pat[['WL_ID_CODE', 'ETHCAT']], on='WL_ID_CODE', how='left')
    pe = pd.to_numeric(eth_join['ETHCAT'], errors='coerce').fillna(-1).astype(int).values
    all_compat.append(sd['compat'])
    all_weights.append(ws)
    all_pair_ethcat.append(pe)
print(f'Done in {time.time()-t0:.1f}s. {len(all_compat)} sims in memory.')

In [ ]:
# RUN ALL 8 CONFIGURATIONS  (4 ethnicities × {min direction, max direction})


runs = {eth: {} for eth in TESTED_ETHCATS}
t_all = time.time()

for eth in TESTED_ETHCATS:
    for direction, v_value in PERTURBATIONS.items():
        
        mults = {e: 1.0 for e in ETHCATS}
        mults[eth] = v_value

        print(f"\n=== Boosting {ETH_LABELS[eth]:13s} direction={direction:3s} (v={v_value})  →  expected to push F({ETH_LABELS[eth]}) {'DOWN' if direction=='min' else 'UP'} ===")
        t_run = time.time()
        sim_results = []
        for sim_id in range(N_SIMS):
            res = run_simulation_maxtx(
                sim_id,
                all_compat[sim_id], all_weights[sim_id], all_pair_ethcat[sim_id],
                mults, SIM_PARAMS)
            sim_results.append(res)
        runs[eth][direction] = sim_results

        mean_F_eth = np.mean([r['F_per_eth'][eth] for r in sim_results])
        mean_F_pob = np.mean([r['F_total'] for r in sim_results])
        print(f"   mean F({ETH_LABELS[eth]}) = {mean_F_eth:.4f}   |   mean F_pob = {mean_F_pob:.4f}   |   {(time.time()-t_run)/60:.1f} min")

print(f"\n>>> All 8 configurations done in {(time.time()-t_all)/60:.1f} min")

In [ ]:
# AGGREGATE INTO THE BOUNDS TABLE


def mean_ci(values, conf=0.95):
    arr = np.asarray([v for v in values if pd.notna(v)], dtype=float)
    if len(arr) < 2:
        if len(arr) == 1: return arr[0], f"{arr[0]:.3f} [-; -]"
        return float('nan'), 'nan'
    m, s = arr.mean(), arr.std(ddof=1)
    low, high = st.t.interval(conf, len(arr)-1, loc=m, scale=s/np.sqrt(len(arr)))
    return m, f"{m:.3f} [{low:.3f}; {high:.3f}]"

def _mean_hla_eth(rs, key, eth):
    vals = [np.mean(r['quality'][(key, eth)]) for r in rs if r['quality'][(key, eth)]]
    return mean_ci(vals)[1]

def _mean_wt_eth(rs, eth):
    vals = [np.mean(r['waiting_times_by_eth'][eth]) for r in rs if r['waiting_times_by_eth'][eth]]
    return mean_ci(vals)[1]

def _mean_pool_eth(rs, eth):
    vals = [r['avg_pool_size_by_eth'][eth] for r in rs]
    return mean_ci(vals)[1]


rows = []
for eth in TESTED_ETHCATS:
    rs_min = runs[eth]['min']
    rs_max = runs[eth]['max']

    arr_eth = np.mean([r['arrivals_by_eth'][eth] for r in rs_min])
    F_min_vals = [r['F_per_eth'][eth] for r in rs_min]
    F_max_vals = [r['F_per_eth'][eth] for r in rs_max]
    m_Fmin, txt_Fmin = mean_ci(F_min_vals)
    m_Fmax, txt_Fmax = mean_ci(F_max_vals)
    delta_F = m_Fmax - m_Fmin

    L_min_vals = [r['L_per_eth'][eth] for r in rs_min]
    L_max_vals = [r['L_per_eth'][eth] for r in rs_max]
    _, txt_Lmin = mean_ci(L_min_vals)
    _, txt_Lmax = mean_ci(L_max_vals)

    rows.append({
        'Ethnicity':           ETH_LABELS[eth],
        'Arrivals':            round(arr_eth, 2),
        'F_min(s)':            txt_Fmin,
        'F_max(s)':            txt_Fmax,
        'ΔF = F_max - F_min':  round(delta_F, 4),
        'HLA Antigen @F_min':  _mean_hla_eth(rs_min, 'antigen', eth),
        'HLA Antigen @F_max':  _mean_hla_eth(rs_max, 'antigen', eth),
        'HLA Allele @F_min':   _mean_hla_eth(rs_min, 'allele',  eth),
        'HLA Allele @F_max':   _mean_hla_eth(rs_max, 'allele',  eth),
        'HLA Eplets @F_min':   _mean_hla_eth(rs_min, 'eplet',   eth),
        'HLA Eplets @F_max':   _mean_hla_eth(rs_max, 'eplet',   eth),
        'L(s) @F_min':         txt_Lmin,
        'L(s) @F_max':         txt_Lmax,
        'Waiting Time @F_min': _mean_wt_eth(rs_min, eth),
        'Waiting Time @F_max': _mean_wt_eth(rs_max, eth),
        'Pool Size @F_min':    _mean_pool_eth(rs_min, eth),
        'Pool Size @F_max':    _mean_pool_eth(rs_max, eth),
    })


configs_all = [(eth, direction) for eth in TESTED_ETHCATS for direction in ('min', 'max')]

def _per_config_mean(metric_fn):
   
    return [np.mean([metric_fn(r) for r in runs[eth][direction]]) for eth, direction in configs_all]

def _per_config_pooled_quality(key):
   
    out = []
    for eth, direction in configs_all:
        per_sim = []
        for r in runs[eth][direction]:
            pooled = [v for e in TESTED_ETHCATS for v in r['quality'][(key, e)]]
            if pooled: per_sim.append(np.mean(pooled))
        out.append(np.mean(per_sim) if per_sim else float('nan'))
    return out

def _per_config_pooled_wt():
    out = []
    for eth, direction in configs_all:
        per_sim = []
        for r in runs[eth][direction]:
            pooled = [w for e in TESTED_ETHCATS for w in r['waiting_times_by_eth'][e]]
            if pooled: per_sim.append(np.mean(pooled))
        out.append(np.mean(per_sim) if per_sim else float('nan'))
    return out



def _F_pob_excl_minorities(r):
    
    arr_main = sum(r['arrivals_by_eth'].get(e, 0) for e in TESTED_ETHCATS)
    if arr_main == 0: return float('nan')
    tx_main = sum(r['F_per_eth'][e] * r['arrivals_by_eth'].get(e, 0) for e in TESTED_ETHCATS)
    return tx_main / arr_main

def _L_pob_excl_minorities(r):
    
    arr_main = sum(r['arrivals_by_eth'].get(e, 0) for e in TESTED_ETHCATS)
    if arr_main == 0: return float('nan')
    dep_main = sum(r['L_per_eth'][e] * r['arrivals_by_eth'].get(e, 0) for e in TESTED_ETHCATS)
    return dep_main / arr_main

def _arr_excl_minorities(r):
    
    return sum(r['arrivals_by_eth'].get(e, 0) for e in TESTED_ETHCATS)

F_pob_pc   = _per_config_mean(_F_pob_excl_minorities)
L_pob_pc   = _per_config_mean(_L_pob_excl_minorities)
arr_pc     = _per_config_mean(_arr_excl_minorities)
HLA_ant_pc = _per_config_pooled_quality('antigen')
HLA_all_pc = _per_config_pooled_quality('allele')
HLA_epl_pc = _per_config_pooled_quality('eplet')
WT_pc      = _per_config_pooled_wt()
Pool_pc    = _per_config_mean(lambda r: r['avg_pool_size'])


def _split_by_dir(per_config_list):
    
    mins = [v for v, (e, d) in zip(per_config_list, configs_all) if d == 'min']
    maxs = [v for v, (e, d) in zip(per_config_list, configs_all) if d == 'max']
    return mins, maxs

def _ci_text(values):
    
    arr = np.asarray([v for v in values if pd.notna(v)], dtype=float)
    if len(arr) < 2:
        return f'{arr[0]:.4f}' if len(arr) == 1 else 'nan'
    m, s = arr.mean(), arr.std(ddof=1)
    low, high = st.t.interval(0.95, len(arr)-1, loc=m, scale=s/np.sqrt(len(arr)))
    return f'{m:.4f} [{low:.4f}; {high:.4f}]'


F_min_dir, F_max_dir = _split_by_dir(F_pob_pc)
L_min_dir, L_max_dir = _split_by_dir(L_pob_pc)
HLAant_min_dir, HLAant_max_dir = _split_by_dir(HLA_ant_pc)
HLAall_min_dir, HLAall_max_dir = _split_by_dir(HLA_all_pc)
HLAepl_min_dir, HLAepl_max_dir = _split_by_dir(HLA_epl_pc)
WT_min_dir, WT_max_dir = _split_by_dir(WT_pc)
Pool_min_dir, Pool_max_dir = _split_by_dir(Pool_pc)


delta_F_pop = float(np.mean(F_max_dir) - np.mean(F_min_dir))

rows.append({
    'Ethnicity':           'Entire Population',
    'Arrivals':            round(np.mean(arr_pc), 2),
    'F_min(s)':            _ci_text(F_min_dir),
    'F_max(s)':            _ci_text(F_max_dir),
    'ΔF = F_max - F_min':  round(delta_F_pop, 5),
    'HLA Antigen @F_min':  _ci_text(HLAant_min_dir),
    'HLA Antigen @F_max':  _ci_text(HLAant_max_dir),
    'HLA Allele @F_min':   _ci_text(HLAall_min_dir),
    'HLA Allele @F_max':   _ci_text(HLAall_max_dir),
    'HLA Eplets @F_min':   _ci_text(HLAepl_min_dir),
    'HLA Eplets @F_max':   _ci_text(HLAepl_max_dir),
    'L(s) @F_min':         _ci_text(L_min_dir),
    'L(s) @F_max':         _ci_text(L_max_dir),
    'Waiting Time @F_min': _ci_text(WT_min_dir),
    'Waiting Time @F_max': _ci_text(WT_max_dir),
    'Pool Size @F_min':    _ci_text(Pool_min_dir),
    'Pool Size @F_max':    _ci_text(Pool_max_dir),
})

bounds_table = pd.DataFrame(rows)
print('\n========== MaxTransplant F(s) BOUNDS TABLE ==========\n')
display(bounds_table)


print('\n--- F_pob per configuration (8 configs, EXCLUDING AmInd/PacIsl) ---')
for (eth, direction), F_pob in zip(configs_all, F_pob_pc):
    print(f'  ({ETH_LABELS[eth]:13s}, {direction:3s}, v={"0.9999" if direction=="min" else "1.0001"}): mean F_pob = {F_pob:.5f}')
print(f'\n  range: [{min(F_pob_pc):.5f}, {max(F_pob_pc):.5f}]  ΔF_pob = {max(F_pob_pc) - min(F_pob_pc):.5f}')


out_path = RESULTS_DIR / 'maxtransplant_bounds_10loci.xlsx'
with pd.ExcelWriter(out_path) as writer:
    bounds_table.to_excel(writer, sheet_name='maxtx_bounds', index=False)
print(f'\nSaved: {out_path}')

In [ ]:

from scipy.stats import wilcoxon, binomtest

def _paired_rank_tests(diffs):
    diffs = np.asarray(diffs, dtype=float)
    diffs = diffs[~np.isnan(diffs)]
    non_zero = diffs[diffs != 0]
    if len(non_zero) > 0:
        try: _, p_w = wilcoxon(non_zero, alternative='two-sided')
        except ValueError: p_w = float('nan')
    else:
        p_w = float('nan')
    n_pos = int((diffs > 0).sum()); n_neg = int((diffs < 0).sum())
    n_tot = n_pos + n_neg
    p_s = binomtest(n_pos, n_tot, 0.5, alternative='two-sided').pvalue if n_tot > 0 else float('nan')
    return {
        'p_wilcoxon': p_w, 'p_sign': p_s,
        'n_pos': n_pos, 'n_neg': n_neg, 'n_used': int(len(diffs)),
        'median_diff': float(np.median(diffs)) if len(diffs) else float('nan'),
    }

def _per_sim_metric(rs, eth, metric):
    
    if metric == 'F(s) (Matched)':
        return [r['F_per_eth'][eth] for r in rs]
    if metric == 'L(s) (Left Unmatched)':
        return [r['L_per_eth'][eth] for r in rs]
    if metric == 'Waiting Time':
        return [np.mean(r['waiting_times_by_eth'][eth]) if r['waiting_times_by_eth'][eth] else float('nan') for r in rs]
   
    key_map = {'HLA Antigen': 'antigen', 'HLA Allele': 'allele', 'HLA Eplets': 'eplet'}
    if metric in key_map:
        k = key_map[metric]
        return [np.mean(r['quality'][(k, eth)]) if r['quality'][(k, eth)] else float('nan') for r in rs]
    raise ValueError(f'Unknown metric {metric}')

METRICS_TO_TEST = [
    'F(s) (Matched)',
    'L(s) (Left Unmatched)',
    'HLA Antigen',
    'HLA Allele',
    'HLA Eplets',
    'Waiting Time',
]


bounds_sig = {}
for eth in TESTED_ETHCATS:
    bounds_sig[eth] = {}
    rs_min = runs[eth]['min']
    rs_max = runs[eth]['max']
    for metric in METRICS_TO_TEST:
        v_min = np.array(_per_sim_metric(rs_min, eth, metric), dtype=float)
        v_max = np.array(_per_sim_metric(rs_max, eth, metric), dtype=float)
        mask = ~(np.isnan(v_min) | np.isnan(v_max))
        diffs = v_max[mask] - v_min[mask]
        bounds_sig[eth][metric] = _paired_rank_tests(diffs)

def _flags(r):
    f = ''
    if not np.isnan(r['p_wilcoxon']) and r['p_wilcoxon'] < 0.05: f += 'W'
    if not np.isnan(r['p_sign'])     and r['p_sign']     < 0.05: f += 'S'
    return f


print('===== F_max - F_min paired significance flags  [W = Wilcoxon p<.05, S = sign test p<.05] =====')
header = f"{'Metric':24s}" + "".join(f"{ETH_LABELS[e]:>14s}" for e in TESTED_ETHCATS)
print(header); print('-' * len(header))
for m in METRICS_TO_TEST:
    row = f"{m:24s}"
    for e in TESTED_ETHCATS:
        row += f"{('[' + _flags(bounds_sig[e][m]) + ']'):>14s}"
    print(row)


print('\n===== Detailed F_max - F_min stats (median diff, n+ / n-, p-values) =====')
print(f"{'Metric':24s}  {'Ethnicity':14s}  {'med Δ':>8s}  {'n+':>4s}  {'n-':>4s}  {'p_W':>9s}  {'p_S':>9s}")
for m in METRICS_TO_TEST:
    for e in TESTED_ETHCATS:
        r = bounds_sig[e][m]
        print(f"{m:24s}  {ETH_LABELS[e]:14s}  {r['median_diff']:>+8.4f}  "
              f"{r['n_pos']:>4d}  {r['n_neg']:>4d}  {r['p_wilcoxon']:>9.4f}  {r['p_sign']:>9.4f}")
    print()

In [ ]:
# SAVE PUBLICATION STYLE EXCEL WITH SIGNIFICANCE FORMATTING 

from openpyxl import Workbook
from openpyxl.styles import Font

ALPHA = 0.05
LABEL_TO_CODE = {ETH_LABELS[e]: e for e in TESTED_ETHCATS}


COL_TO_METRIC = {
    'F_min(s)':            'F(s) (Matched)',
    'F_max(s)':            'F(s) (Matched)',
    'L(s) @F_min':         'L(s) (Left Unmatched)',
    'L(s) @F_max':         'L(s) (Left Unmatched)',
    'HLA Antigen @F_min':  'HLA Antigen',
    'HLA Antigen @F_max':  'HLA Antigen',
    'HLA Allele @F_min':   'HLA Allele',
    'HLA Allele @F_max':   'HLA Allele',
    'HLA Eplets @F_min':   'HLA Eplets',
    'HLA Eplets @F_max':   'HLA Eplets',
    'Waiting Time @F_min': 'Waiting Time',
    'Waiting Time @F_max': 'Waiting Time',
}

wb = Workbook()
ws = wb.active
ws.title = 'maxtx_bounds'


for col_idx, col_name in enumerate(bounds_table.columns, start=1):
    c = ws.cell(row=1, column=col_idx, value=col_name)
    c.font = Font(bold=True)


for row_pos, (_, row) in enumerate(bounds_table.iterrows(), start=2):
    eth_label = row['Ethnicity']
    eth_code = LABEL_TO_CODE.get(eth_label)  

    for col_idx, col_name in enumerate(bounds_table.columns, start=1):
        val = row[col_name]
        cell = ws.cell(row=row_pos, column=col_idx, value=val)

        
        if col_name in COL_TO_METRIC and eth_code in TESTED_ETHCATS:
            metric = COL_TO_METRIC[col_name]
            r = bounds_sig[eth_code][metric]
            bold      = (not np.isnan(r['p_wilcoxon'])) and r['p_wilcoxon'] < ALPHA
            underline = (not np.isnan(r['p_sign']))     and r['p_sign']     < ALPHA
            if bold or underline:
                cell.font = Font(bold=bold, underline='single' if underline else None)


for col_idx, col_name in enumerate(bounds_table.columns, start=1):
    col_letter = chr(64 + col_idx) if col_idx <= 26 else 'A' + chr(64 + col_idx - 26)
    ws.column_dimensions[col_letter].width = max(14, len(col_name) + 2)


foot = len(bounds_table) + 4
ws.cell(row=foot, column=1, value='Significance flags — paired test F_max(s, sim) - F_min(s, sim) across 100 sims:')
fb = ws.cell(row=foot+1, column=1, value='   bold       = Wilcoxon signed-rank p < 0.05 (paired bound, primary)')
fu = ws.cell(row=foot+2, column=1, value='   underlined = sign test p < 0.05 (paired bound, robustness)')
ws.cell(row=foot+3, column=1, value='   Entire Population row: shows the actual range of each metric across the 8 configurations (4 eth × 2 directions); no paired test is reported because there is no natural pairing for the aggregate.')
fb.font = Font(bold=True); fu.font = Font(underline='single')

out_path = RESULTS_DIR / 'maxtransplant_bounds_10loci_significance.xlsx'
wb.save(out_path)
print(f'Saved: {out_path}')